## QAOA-подходы в этом ноутбуке и наш QAOA-inspired метод

В этом ноутбуке рассматривается несколько различных реализаций QAOA-подходов к решению одной и той же QUBO/Ising-задачи. Под QAOA здесь понимается не один конкретный алгоритм, а семейство методов, которые сохраняют общую идею: слоистую вариационную структуру с чередованием **cost**- и **mixer**-операторов и настраиваемыми параметрами $\{\gamma_l\}, \{\beta_l\}$. На практике это может включать как настоящие гейтовые квантовые схемы, так и полностью классические эвристики, вдохновлённые архитектурой QAOA.

Один из таких вариантов, реализованный в этом ноутбуке, — классический **QAOA-inspired** метод, работающий полностью на CPU. Он не использует квантовые состояния и квантовые гейты напрямую, но сохраняет архитектурную логику QAOA: глубину $p$, последовательности параметров $\gamma_l$ и $\beta_l$, фиксированный cost Hamiltonian и отдельный механизм перемешивания. Поэтому этот метод удобно рассматривать как одну из возможных реализаций QAOA-подхода наряду с более близкими к квантовым схемам вариантами.

### Единая постановка задачи

Для всех рассматриваемых в ноутбуке QAOA-подходов используется одна и та же Ising-постановка:

$$
E_{\mathrm{Ising}}(s) = \mathrm{const} + \sum_i h_i s_i + \sum_{i<j} J_{ij} s_i s_j
$$

где $s_i \in \{-1, 1\}$ — спиновые переменные. Это позволяет сравнивать разные реализации на одной и той же задаче: меняется не сама функция энергии, а способ исследования пространства решений.

### Как устроен наш QAOA-inspired вариант

В данной реализации вместо квантового состояния используется популяция классических спиновых конфигураций $s^{(k)} \in \{-1,1\}^n$. Алгоритм строится как последовательность из $p$ слоёв, и каждый слой состоит из двух шагов:

- **cost step** с параметром $\gamma_l$;
- **mixer step** с параметром $\beta_l$.

На шаге cost для каждой конфигурации вычисляется энергия $E(s^{(k)})$, после чего конфигурации перевзвешиваются в пользу низкоэнергетических состояний. Это делается с помощью экспоненциальных весов вида

$$
w_k \propto \exp\left(-\gamma_l \bigl(E(s^{(k)}) - E_{\min}\bigr)\right)
$$

где $E_{\min}$ — минимальная энергия в текущей популяции. Затем выполняется resampling, и конфигурации с лучшей энергией получают большее представительство в следующем поколении.

На шаге mixer выполняется стохастическое перемешивание популяции через локальные flips спинов. Для каждой конфигурации пробуются изменения вида $s_i \rightarrow -s_i$, и эти изменения принимаются либо всегда при улучшении энергии, либо с некоторой вероятностью при ухудшении. Тем самым микшерный слой помогает алгоритму не застревать слишком рано в локальных минимумах.

### Как оценивается результат

После завершения всех слоёв выбирается лучшая найденная спиновая конфигурация. Затем она декодируется обратно в бинарное QUBO-решение и далее в расписание показов.

Для оценки качества используются те же прикладные метрики, что и для других solvers в ноутбуке:

- суммарная predicted revenue;
- число выбранных показов;
- число физических конфликтов в одном зале;
- число конфликтов типа same-movie cross-hall.

Это позволяет сравнивать данный QAOA-inspired подход с другими реализациями QAOA и с классическими baseline-методами в единой постановке.


In [10]:
from __future__ import annotations

from pathlib import Path
import json
from typing import Dict, Tuple, List
from scipy.optimize import minimize

import numpy as np
import pandas as pd


PROJECT_ROOT = Path("/Users/mishatrubik/Desktop/QC/BOQ")
DATA_DIR = PROJECT_ROOT / "data"
QUBO_DIR = DATA_DIR / "qubo"

QUBO_MATRIX_PATH = QUBO_DIR / "universal_qubo_matrix.json"
VAR_INDEX_MAP_PATH = QUBO_DIR / "universal_qubo_var_index_map.json"
CANDIDATE_PATH = QUBO_DIR / "universal_candidate_scored.csv"


# --------------------------------------------------
# Load saved QUBO artifacts
# --------------------------------------------------
with QUBO_MATRIX_PATH.open("r", encoding="utf-8") as f:
    qubo_Q_raw = json.load(f)

with VAR_INDEX_MAP_PATH.open("r", encoding="utf-8") as f:
    qubo_var_index_map = json.load(f)

qubo_var_index_map_int = {int(k): int(v) for k, v in qubo_var_index_map.items()}

universal_candidate_scored_df = pd.read_csv(
    CANDIDATE_PATH,
    parse_dates=["show_datetime", "end_datetime"],
)

num_variables = len(qubo_var_index_map_int)


# --------------------------------------------------
# Build dense Q matrix from sparse terms
# JSON format:
# [
#   {"i": 0, "j": 0, "q": ...},
#   ...
# ]
# --------------------------------------------------
Q_mat = np.zeros((num_variables, num_variables), dtype=float)

for term in qubo_Q_raw:
    i = int(term["i"])
    j = int(term["j"])
    q = float(term["q"])

    Q_mat[i, j] += q
    if i != j:
        Q_mat[j, i] += q


# --------------------------------------------------
# QUBO -> Ising conversion
# E_qubo(x) = sum_i Q_ii x_i + sum_{i<j} Q_ij x_i x_j
# x_i = (1 + s_i) / 2
# --------------------------------------------------
def qubo_to_ising(Q: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
    n = Q.shape[0]
    h = np.zeros(n, dtype=float)
    J = np.zeros((n, n), dtype=float)
    const = 0.0

    for i in range(n):
        h[i] += Q[i, i] / 2.0
        const += Q[i, i] / 2.0

    for i in range(n):
        for j in range(i + 1, n):
            q = Q[i, j]
            if q != 0.0:
                J[i, j] += q / 4.0
                J[j, i] += q / 4.0
                h[i] += q / 4.0
                h[j] += q / 4.0
                const += q / 4.0

    return h, J, const


h_vec, J_mat, qubo_const = qubo_to_ising(Q_mat)


# --------------------------------------------------
# Energy / decoding / evaluation helpers
# --------------------------------------------------
def ising_energy(
    spins: np.ndarray,
    h_vec: np.ndarray,
    J_mat: np.ndarray,
    const: float = 0.0,
) -> float:
    spins = np.asarray(spins, dtype=float)
    e = const + np.dot(h_vec, spins)

    n = len(spins)
    for i in range(n):
        for j in range(i + 1, n):
            e += J_mat[i, j] * spins[i] * spins[j]

    return float(e)


def spins_to_binary_sample(state):
    state = np.asarray(state, dtype=int)
    x = (state + 1) // 2
    return {i: int(x[i]) for i in range(len(x))}


def decode_sample_to_schedule(sample, var_index_to_candidate_idx, candidate_df):
    selected_cand_idx = [
        var_index_to_candidate_idx[v]
        for v, bit in sample.items()
        if int(bit) == 1
    ]
    schedule = candidate_df.loc[selected_cand_idx].copy()
    return schedule


def intervals_overlap(start_a, end_a, start_b, end_b):
    return max(start_a, start_b) < min(end_a, end_b)


def find_physical_hall_conflicts(schedule_df):
    if len(schedule_df) == 0:
        return pd.DataFrame()

    work_df = schedule_df.copy().sort_values(
        ["cinema_id", "hall_id", "show_datetime"]
    )

    end_col = "end_datetime" 

    conflicts = []

    for (cinema_id, hall_id), group in work_df.groupby(["cinema_id", "hall_id"]):
        rows = list(group.itertuples())

        for i in range(len(rows)):
            a = rows[i]
            for j in range(i + 1, len(rows)):
                b = rows[j]

                a_end = getattr(a, end_col)
                b_end = getattr(b, end_col)

                if a_end <= b.show_datetime:
                    break

                if intervals_overlap(
                    a.show_datetime,
                    a_end,
                    b.show_datetime,
                    b_end,
                ):
                    conflicts.append(
                        {
                            "cinema_id": cinema_id,
                            "hall_id": hall_id,
                            "candidate_idx_a": a.Index,
                            "candidate_idx_b": b.Index,
                            "movie_id_a": a.movie_id,
                            "movie_id_b": b.movie_id,
                            "start_a": a.show_datetime,
                            "end_a": a_end,
                            "start_b": b.show_datetime,
                            "end_b": b_end,
                            "pred_revenue_a": a.pred_revenue,
                            "pred_revenue_b": b.pred_revenue,
                        }
                    )

    return pd.DataFrame(conflicts)


def find_same_movie_cross_hall_conflicts(schedule_df):
    if len(schedule_df) == 0:
        return pd.DataFrame()

    work_df = schedule_df.copy().sort_values(
        ["cinema_id", "movie_id", "show_datetime"]
    )

    end_col = "end_datetime" 

    conflicts = []

    for (cinema_id, movie_id), group in work_df.groupby(["cinema_id", "movie_id"]):
        rows = list(group.itertuples())

        for i in range(len(rows)):
            a = rows[i]
            for j in range(i + 1, len(rows)):
                b = rows[j]

                a_end = getattr(a, end_col)
                b_end = getattr(b, end_col)

                if a_end <= b.show_datetime:
                    break

                if a.hall_id == b.hall_id:
                    continue

                if intervals_overlap(
                    a.show_datetime,
                    a_end,
                    b.show_datetime,
                    b_end,
                ):
                    conflicts.append(
                        {
                            "cinema_id": cinema_id,
                            "movie_id": movie_id,
                            "candidate_idx_a": a.Index,
                            "candidate_idx_b": b.Index,
                            "hall_id_a": a.hall_id,
                            "hall_id_b": b.hall_id,
                            "start_a": a.show_datetime,
                            "end_a": a_end,
                            "start_b": b.show_datetime,
                            "end_b": b_end,
                            "pred_revenue_a": a.pred_revenue,
                            "pred_revenue_b": b.pred_revenue,
                        }
                    )

    return pd.DataFrame(conflicts)


def evaluate_schedule(name, spins, solver_energy):
    spins = np.asarray(spins, dtype=int)

    assert len(spins) == num_variables, (
        f"Spin vector length {len(spins)} != num_variables {num_variables}"
    )

    sample = spins_to_binary_sample(spins)
    schedule = decode_sample_to_schedule(
        sample,
        qubo_var_index_map_int,
        universal_candidate_scored_df,
    )

    reconstructed_ising_energy = ising_energy(
        spins,
        h_vec,
        J_mat,
        qubo_const,
    )

    hall_conflicts_df = find_physical_hall_conflicts(schedule)
    same_movie_conflicts_df = find_same_movie_cross_hall_conflicts(schedule)

    metrics = {
        "solver": name,
        "solver_energy": float(solver_energy),
        "reconstructed_ising_energy": float(reconstructed_ising_energy),
        "shows": int(len(schedule)),
        "revenue": float(schedule["pred_revenue"].sum()) if len(schedule) > 0 else 0.0,
        "hall_conflicts": int(len(hall_conflicts_df)),
        "same_movie_conflicts": int(len(same_movie_conflicts_df)),
    }

    return schedule, hall_conflicts_df, same_movie_conflicts_df, metrics


print("Loaded QUBO matrix from:", QUBO_MATRIX_PATH)
print("Loaded candidate df from:", CANDIDATE_PATH)
print("num_variables:", num_variables)
print("Q_mat shape:", Q_mat.shape)
print("h_vec shape:", h_vec.shape)
print("J_mat shape:", J_mat.shape)
print("qubo_const:", qubo_const)


Loaded QUBO matrix from: /Users/mishatrubik/Desktop/QC/BOQ/data/qubo/universal_qubo_matrix.json
Loaded candidate df from: /Users/mishatrubik/Desktop/QC/BOQ/data/qubo/universal_candidate_scored.csv
num_variables: 32
Q_mat shape: (32, 32)
h_vec shape: (32,)
J_mat shape: (32, 32)
qubo_const: 10218771.93


In [11]:


def compute_delta_energy_single_flip(spins, i, h_vec, J_mat):

    local_field = h_vec[i] + np.dot(J_mat[i], spins) - J_mat[i, i] * spins[i]
    delta_e = -2.0 * spins[i] * local_field
    return float(delta_e)


def initialize_population(num_variables, population_size, seed=42):
    rng = np.random.default_rng(seed)
    population = rng.choice([-1, 1], size=(population_size, num_variables))
    return population


def population_energies(population, h_vec, J_mat, qubo_const):
    energies = np.array(
        [ising_energy(spins, h_vec, J_mat, qubo_const) for spins in population],
        dtype=float,
    )
    return energies


def qaoa_cost_resample(population, energies, gamma, rng):
    """
    QAOA-inspired 'cost step':
    усиливаем хорошие конфигурации через веса exp(-gamma * shifted_energy)
    и пересэмплируем популяцию.
    """
    shifted = energies - energies.min()
    weights = np.exp(-gamma * shifted)

    if not np.all(np.isfinite(weights)) or weights.sum() == 0:
        weights = np.ones_like(weights) / len(weights)
    else:
        weights = weights / weights.sum()

    sampled_idx = rng.choice(len(population), size=len(population), replace=True, p=weights)
    new_population = population[sampled_idx].copy()
    return new_population, weights


def qaoa_mixer_step(population, beta, h_vec, J_mat, rng):
    """
    QAOA-inspired 'mixer step':
    для каждой конфигурации пытаемся флипать спины.
    beta интерпретируем как интенсивность перемешивания.
    Чем больше beta, тем больше попыток локального движения.
    """
    pop = population.copy()
    num_vars = pop.shape[1]

    flip_prob = min(max(beta, 0.0), 1.0)

    for k in range(len(pop)):
        for i in range(num_vars):
            if rng.random() < flip_prob:
                delta_e = compute_delta_energy_single_flip(pop[k], i, h_vec, J_mat)

                # Мягкий biased move:
                # хорошие флипы принимаем всегда,
                # плохие — с вероятностью exp(-beta * delta_e)
                if delta_e <= 0:
                    pop[k, i] *= -1
                else:
                    accept_prob = np.exp(-beta * delta_e)
                    if rng.random() < accept_prob:
                        pop[k, i] *= -1

    return pop


def run_qaoa_inspired(
    h_vec,
    J_mat,
    qubo_const,
    depth_p=5,
    population_size=64,
    gamma_schedule=None,
    beta_schedule=None,
    seed=42,
):
    """
    Полностью классический QAOA-inspired solver:
    - population of spin configurations
    - layered cost/resample step
    - layered mixer step
    """
    rng = np.random.default_rng(seed)

    if gamma_schedule is None:
        gamma_schedule = np.linspace(0.3, 1.2, depth_p)

    if beta_schedule is None:
        beta_schedule = np.linspace(0.25, 0.05, depth_p)

    gamma_schedule = np.asarray(gamma_schedule, dtype=float)
    beta_schedule = np.asarray(beta_schedule, dtype=float)

    assert len(gamma_schedule) == depth_p
    assert len(beta_schedule) == depth_p

    population = initialize_population(
        num_variables=len(h_vec),
        population_size=population_size,
        seed=seed,
    )

    history = []

    best_state = None
    best_energy = np.inf

    for layer_idx in range(depth_p):
        gamma = float(gamma_schedule[layer_idx])
        beta = float(beta_schedule[layer_idx])

        energies_before = population_energies(population, h_vec, J_mat, qubo_const)

        layer_best_idx = int(np.argmin(energies_before))
        layer_best_energy = float(energies_before[layer_best_idx])

        if layer_best_energy < best_energy:
            best_energy = layer_best_energy
            best_state = population[layer_best_idx].copy()

        population, weights = qaoa_cost_resample(
            population=population,
            energies=energies_before,
            gamma=gamma,
            rng=rng,
        )

        population = qaoa_mixer_step(
            population=population,
            beta=beta,
            h_vec=h_vec,
            J_mat=J_mat,
            rng=rng,
        )

        energies_after = population_energies(population, h_vec, J_mat, qubo_const)

        layer_record = {
            "layer": layer_idx + 1,
            "gamma": gamma,
            "beta": beta,
            "energy_mean_before": float(np.mean(energies_before)),
            "energy_best_before": float(np.min(energies_before)),
            "energy_mean_after": float(np.mean(energies_after)),
            "energy_best_after": float(np.min(energies_after)),
        }
        history.append(layer_record)

        layer_best_idx_after = int(np.argmin(energies_after))
        layer_best_energy_after = float(energies_after[layer_best_idx_after])

        if layer_best_energy_after < best_energy:
            best_energy = layer_best_energy_after
            best_state = population[layer_best_idx_after].copy()

    history_df = pd.DataFrame(history)

    return {
        "best_state": np.asarray(best_state, dtype=int),
        "best_energy": float(best_energy),
        "history_df": history_df,
        "gamma_schedule": gamma_schedule,
        "beta_schedule": beta_schedule,
    }


# --------------------------------------------------
# Run QAOA-inspired solver
# --------------------------------------------------
qaoa_result = run_qaoa_inspired(
    h_vec=h_vec,
    J_mat=J_mat,
    qubo_const=qubo_const,
    depth_p=6,
    population_size=96,
    gamma_schedule=np.linspace(0.4, 1.6, 6),
    beta_schedule=np.linspace(0.22, 0.04, 6),
    seed=42,
)

qaoa_best_state = qaoa_result["best_state"]
qaoa_best_energy = qaoa_result["best_energy"]
qaoa_history_df = qaoa_result["history_df"]

qaoa_schedule, qaoa_hall_conflicts, qaoa_same_movie_conflicts, qaoa_metrics = evaluate_schedule(
    "QAOA-inspired CPU",
    qaoa_best_state,
    qaoa_best_energy,
)

qaoa_schedule_view = (
    qaoa_schedule.sort_values(["cinema_id", "hall_id", "show_datetime"])
    .reset_index()
    .rename(columns={"index": "candidate_idx"})
)

print("QAOA-inspired best energy:", qaoa_best_energy)
print()
display(qaoa_history_df)
print()
display(pd.DataFrame([qaoa_metrics]))
print()
display(qaoa_schedule_view)
print()
display(qaoa_hall_conflicts)
print()
display(qaoa_same_movie_conflicts)


QAOA-inspired best energy: -320748.58999999985



,layer,gamma,beta,energy_mean_before,energy_best_before,energy_mean_after,energy_best_after
0,1,0.40,0.220,1.008204e+07,1637617.92,829868.799375,-320748.59
1,2,0.64,0.184,8.298688e+05,-320748.59,-320748.590000,-320748.59
2,3,0.88,0.148,-3.207486e+05,-320748.59,-320748.590000,-320748.59
3,4,1.12,0.112,-3.207486e+05,-320748.59,-320748.590000,-320748.59
4,5,1.36,0.076,-3.207486e+05,-320748.59,-320748.590000,-320748.59
5,6,1.60,0.040,-3.207486e+05,-320748.59,-320748.590000,-320748.59


,solver,solver_energy,reconstructed_ising_energy,shows,revenue,hall_conflicts,same_movie_conflicts
0,QAOA-inspired CPU,-320748.59,-320748.59,6,320748.59,0,0


,candidate_idx,show_id,slot_id,cinema_id,hall_id,hall_capacity,movie_id,genre,age_rating,runtime_min,...,month,holiday_flag,release_age_days,lead_time_days,prime_time_flag,runtime_bucket,screening_number_for_movie,pred_occupancy_rate,pred_sold_tickets,pred_revenue
0,17,hall_large__s1__movie_dune_2,hall_large__s1,cinema_moscow_toy,hall_large,180,movie_dune_2,sci_fi,16+,166,...,1,False,-45,7.0,False,long,1,0.570182,103,51428.93
1,27,hall_large__s3__movie_horror_01,hall_large__s3,cinema_moscow_toy,hall_large,180,movie_horror_01,horror,18+,98,...,1,False,-49,7.0,True,medium,3,0.908486,164,81996.72
2,28,hall_large__s4__movie_comedy_01,hall_large__s4,cinema_moscow_toy,hall_large,180,movie_comedy_01,comedy,12+,105,...,1,False,-26,7.0,True,medium,4,0.901739,162,80798.31
3,2,hall_small__s1__movie_family_01,hall_small__s1,cinema_moscow_toy,hall_small,90,movie_family_01,animation,6+,92,...,1,False,-36,7.0,False,medium,1,0.535378,48,24030.24
4,10,hall_small__s3__movie_family_01,hall_small__s3,cinema_moscow_toy,hall_small,90,movie_family_01,animation,6+,92,...,1,False,-36,7.0,True,medium,3,0.913910,82,41051.66
5,13,hall_small__s4__movie_dune_2,hall_small__s4,cinema_moscow_toy,hall_small,90,movie_dune_2,sci_fi,16+,166,...,1,False,-45,7.0,True,long,4,0.917141,83,41442.73


""


""


## В этой реализации нет внешнего цикла вариаций параметров $\gamma$ и $\beta$, они берутся статичным массивом для глубины из $p=6$ итераций.
Далее будет реализован цикл с уменьшенной глубиной квантовой части, но с вариациями $\gamma_{ij}$ и $\beta_{ij}$ $i=0,1...n$, $j=0,1..p$. $n$ - глубина внешего цикла, $p$ - глубина квантовой схемы.

## Qiskit QAOA: гейтовая реализация на той же Ising-постановке

После получения корректного решения с помощью классического QAOA-inspired алгоритма следующим шагом является реализация настоящего QAOA на гейтовой квантовой схеме. В этом разделе используется та же самая Ising-постановка задачи, что и выше, однако способ поиска решения меняется: вместо эволюции популяции классических конфигураций строится параметризованная квантовая схема QAOA и проводится вариационная оптимизация её параметров.

В реализации на Qiskit целевая функция кодируется в виде cost Hamiltonian, представленного оператором `SparsePauliOp`. Затем на его основе строится QAOA ansatz глубины $p$, и классический оптимизатор подбирает параметры схемы так, чтобы минимизировать ожидаемую энергию. После завершения оптимизации из финального состояния извлекается наиболее вероятная битовая строка, которая затем декодируется обратно в QUBO-решение и в расписание показов.

Поскольку полная задача может оказаться слишком большой для прямой симуляции квантовой схемы, в этом блоке допускается запуск на ограниченном подмножестве переменных. Это позволяет проверить корректность самой гейтовой реализации QAOA и сравнить её с уже работающим QAOA-inspired подходом в рамках единой постановки.


In [3]:
# Qiskit gate-based QAOA
from __future__ import annotations

import json
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd

from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorSampler

from qiskit_algorithms.minimum_eigensolvers import QAOA
from qiskit_algorithms.optimizers import COBYLA

print("Qiskit imports loaded successfully.")


Qiskit imports loaded successfully.


In [7]:
def build_sparse_pauli_op_from_ising(
    h: np.ndarray,
    J: np.ndarray,
) -> Tuple[SparsePauliOp, float]:
    n = len(h)
    pauli_terms: List[Tuple[str, float]] = []

    for i, coeff in enumerate(h):
        if abs(coeff) > 1e-12:
            label = ["I"] * n
            label[n - 1 - i] = "Z"   
            pauli_terms.append(("".join(label), float(coeff)))

    # Quadratic ZZ terms
    for i in range(n):
        for j in range(i + 1, n):
            coeff = J[i, j]
            if abs(coeff) > 1e-12:
                label = ["I"] * n
                label[n - 1 - i] = "Z"
                label[n - 1 - j] = "Z"
                pauli_terms.append(("".join(label), float(coeff)))

    if len(pauli_terms) == 0:
        op = SparsePauliOp.from_list([("I" * n, 0.0)])
    else:
        op = SparsePauliOp.from_list(pauli_terms)

    return op, float(qubo_const)


cost_operator, constant_shift = build_sparse_pauli_op_from_ising(h_vec, J_mat)

print("Cost operator built.")
print("num_qubits:", len(h_vec))
print("num_pauli_terms:", len(cost_operator))
print("constant_shift:", constant_shift)


Cost operator built.
num_qubits: 32
num_pauli_terms: 166
constant_shift: 10218771.93


In [9]:
MAX_QAOA_VARS = 3   # начни с 8-12, потом можно попробовать больше

active_idx = np.arange(min(MAX_QAOA_VARS, len(h_vec)))

h_small = h_vec[active_idx].copy()
J_small = J_mat[np.ix_(active_idx, active_idx)].copy()

cost_operator_small, constant_shift_small = build_sparse_pauli_op_from_ising(
    h_small,
    J_small,
)
# Optimizer and sampler
optimizer = COBYLA(maxiter=30)
sampler = StatevectorSampler(seed=42)

# QAOA depth
QAOA_REPS = 2

qaoa = QAOA(
    sampler=sampler,
    optimizer=optimizer,
    reps=QAOA_REPS,
)

qaoa_result = qaoa.compute_minimum_eigenvalue(cost_operator_small)

print("QAOA optimization finished.")
print(qaoa_result)



/Users/mishatrubik/venv/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:597: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/mishatrubik/venv/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/Users/mishatrubik/venv/lib/python3.12/site-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


QAOA optimization finished.
{   'aux_operators_evaluated': None,
    'best_measurement': {   'bitstring': '111',
                            'probability': 0.439453125,
                            'state': 7,
                            'value': np.complex128(-1882783.045+0j)},
    'cost_function_evals': 30,
    'eigenstate': {   '000': 0.064453125,
                      '001': 0.015625,
                      '010': 0.1572265625,
                      '011': 0.1865234375,
                      '100': 0.0009765625,
                      '101': 0.064453125,
                      '110': 0.0712890625,
                      '111': 0.439453125},
    'eigenvalue': np.float64(-824636.7437011718),
    'optimal_circuit': <qiskit.circuit.library.n_local.qaoa_ansatz.QAOAAnsatz object at 0x165fbad20>,
    'optimal_parameters': {   ParameterVectorElement(γ[1]): np.float64(-3.123020503186973),
                              ParameterVectorElement(γ[0]): np.float64(-5.572429980664313),
                

## Для большого числа переменных количество итераций матричного решения через QAOA IBM растет экспоненциально и не считается за адекватное время
В коде есть занлушка первыми MAX_QAOA_VARS. 

Такой метод не является работоспособным для реальных задач.

Возможны в перспективе шансы реализовать IBM QAOA на GPU вычислителях тензорных продуктов